In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import json
import matplotlib.pyplot as plt
import os
import shutil
import numpy as np

In [ ]:


SOURCE_DIR = r"C:\Users\WALID\pfe_idsd\image"
BASE_DIR = r"C:\Users\WALID\pfe_idsd"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR = os.path.join(BASE_DIR, "validation")

# Créer les dossiers
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(VAL_DIR, exist_ok=True)

print("🔄 Séparation du dataset en train/validation...")

for class_name in os.listdir(SOURCE_DIR):
    class_path = os.path.join(SOURCE_DIR, class_name)
    
    if not os.path.isdir(class_path):
        continue
    
    # Créer les sous-dossiers
    os.makedirs(os.path.join(TRAIN_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(VAL_DIR, class_name), exist_ok=True)
    
    # Lister les images
    images = [f for f in os.listdir(class_path) 
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.avif'))]
    
    # Mélanger
    np.random.seed(42)
    np.random.shuffle(images)
    
    # Split 80/20
    split_idx = int(len(images) * 0.8)
    train_images = images[:split_idx]
    val_images = images[split_idx:]
    
    # Copier vers train
    for img in train_images:
        src = os.path.join(class_path, img)
        dst = os.path.join(TRAIN_DIR, class_name, img)
        if not os.path.exists(dst):  # Éviter de recopier
            shutil.copy2(src, dst)
    
    # Copier vers validation
    for img in val_images:
        src = os.path.join(class_path, img)
        dst = os.path.join(VAL_DIR, class_name, img)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    
    print(f"✅ {class_name}: {len(train_images)} train, {len(val_images)} val")

print("\n✅ Dataset séparé avec succès !")
print(f"📂 Train : {TRAIN_DIR}")
print(f"📂 Validation : {VAL_DIR}")

🔄 Séparation du dataset en train/validation...
✅ Almoravid_Koubba: 36 train, 9 val
✅ chellah: 40 train, 10 val
✅ Jardin_Majorelle: 36 train, 10 val
✅ kasbah d'agadir oufella: 40 train, 10 val
✅ Kasbah_Amridil: 40 train, 10 val
✅ Kasbah_des_Oudayas: 40 train, 10 val
✅ Kasbah_de_Beni_Mellal: 40 train, 10 val
✅ Kasbah_de_Taourirt: 41 train, 11 val
✅ Ksar_d’Aït_Ben_Haddou: 40 train, 10 val
✅ Mausolee_de_Mohammed V: 40 train, 11 val
✅ menara: 37 train, 10 val
✅ Mosquee_de_la_Kasbah: 36 train, 9 val
✅ Mosquee_Hassan II: 40 train, 10 val
✅ Mosquee_Koutoubia: 40 train, 10 val
✅ Mosquee_Tinmal: 40 train, 10 val
✅ palais royal de fès: 40 train, 10 val
✅ Palais_de_la_Bahia: 36 train, 9 val
✅ Palais_El_Badi: 37 train, 10 val
✅ Place_Jemaa_el-Fna: 36 train, 10 val
✅ Tombeaux_Saadiens: 36 train, 10 val
✅ Volubilis meknès: 40 train, 10 val

✅ Dataset séparé avec succès !
📂 Train : C:\Users\WALID\pfe_idsd\train
📂 Validation : C:\Users\WALID\pfe_idsd\validation


In [5]:
# Chemins
TRAIN_PATH = r"C:\Users\WALID\pfe_idsd\train"
VAL_PATH = r"C:\Users\WALID\pfe_idsd\validation"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # Réduit de 64 à 32
EPOCHS = 50

In [ ]:
#  TRAIN : Data Augmentation MODÉRÉE (pas trop agressive)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)

#  VALIDATION : Seulement rescale
val_datagen = ImageDataGenerator(rescale=1./255)

#  CHARGER
train_data = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

print(f"\n✅ Training : {train_data.samples} images")
print(f"✅ Validation : {val_data.samples} images")

Found 804 images belonging to 21 classes.
Found 205 images belonging to 21 classes.

✅ Training : 804 images
✅ Validation : 205 images


In [7]:
# ✅ SAUVEGARDER LES CLASSES
class_names = list(train_data.class_indices.keys())
with open("class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f)

# ✅ MOBILENETV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

# ✅ ARCHITECTURE AVEC DROPOUT RÉDUIT
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)

# Couche 1 - Dropout réduit à 0.3
x = layers.Dense(512, activation="relu", kernel_regularizer=l2(0.00001))(x)  # ← L2 réduit
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)  # ← RÉDUIT de 0.5 à 0.3

# Couche 2 - Dropout réduit à 0.3
x = layers.Dense(256, activation="relu", kernel_regularizer=l2(0.00001))(x)  # ← L2 réduit
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)  # ← RÉDUIT de 0.5 à 0.3

# Couche 3 - Dropout réduit à 0.2
x = layers.Dense(128, activation="relu", kernel_regularizer=l2(0.00001))(x)  # ← L2 réduit
x = layers.Dropout(0.2)(x)  # ← RÉDUIT de 0.4 à 0.2

# Sortie
output = layers.Dense(train_data.num_classes, activation="softmax")(x)

model = models.Model(inputs=base_model.input, outputs=output)

# ✅ COMPILATION avec LR plus élevé
model.compile(
    optimizer=Adam(learning_rate=0.0005),  # ← AUGMENTÉ de 0.0001 à 0.0005
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\n✅ Modèle créé (régularisation modérée) !")
model.summary()


✅ Modèle créé (régularisation modérée) !


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 3,083,861 (11.76 MB)

 Trainable params: 824,341 (3.14 MB)

 Non-trainable params: 2,259,520 (8.62 MB)

In [8]:
# CALLBACKS
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=20,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        mode='max',
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_phase1_v2.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# ENTRAÎNEMENT
print("PHASE 1 (Version 2) : Avec regularisation moderee...\n")

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# RÉSULTATS
best_val_acc = max(history.history['val_accuracy'])
best_train_acc = max(history.history['accuracy'])
gap = best_train_acc - best_val_acc

print(f"\n{'='*60}")
print(f"PHASE 1 TERMINEE (Version 2)")
print(f"{'='*60}")
print(f"Meilleure train accuracy : {best_train_acc*100:.2f}%")
print(f"Meilleure val accuracy   : {best_val_acc*100:.2f}%")
print(f"Gap train-val            : {gap*100:.2f}%")
print(f"{'='*60}")

PHASE 1 (Version 2) : Avec regularisation moderee...

Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 992ms/step - accuracy: 0.0617 - loss: 3.8944
Epoch 1: val_accuracy improved from None to 0.19512, saving model to best_model_phase1_v2.h5



Epoch 1: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.0958 - loss: 3.5271 - val_accuracy: 0.1951 - val_loss: 2.7895 - learning_rate: 5.0000e-04
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3099 - loss: 2.5287
Epoch 2: val_accuracy improved from 0.19512 to 0.49756, saving model to best_model_phase1_v2.h5



Epoch 2: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step - accuracy: 0.3234 - loss: 2.3896 - val_accuracy: 0.4976 - val_loss: 2.3324 - learning_rate: 5.0000e-04
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 944ms/step - accuracy: 0.4321 - loss: 2.0035
Epoch 3: val_accuracy improved from 0.49756 to 0.56585, saving model to best_model_phase1_v2.h5



Epoch 3: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.4366 - loss: 1.9561 - val_accuracy: 0.5659 - val_loss: 2.0011 - learning_rate: 5.0000e-04
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 907ms/step - accuracy: 0.4921 - loss: 1.7231
Epoch 4: val_accuracy did not improve from 0.56585
26/26 ━━━━━━━━━━━━━━━━━━━━ 29s 1s/step - accuracy: 0.5062 - loss: 1.6751 - val_accuracy: 0.5610 - val_loss: 1.7128 - learning_rate: 5.0000e-04
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5667 - loss: 1.4925
Epoch 5: val_accuracy did not improve from 0.56585
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.5460 - loss: 1.5103 - val_accuracy: 0.5659 - val_loss: 1.5424 - learning_rate: 5.0000e-04
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 924ms/step - accuracy: 0.6065 - loss: 1.3242
Epoch 6: val_accuracy improved from 0.56585 to 0.62439, saving model to best_model_phase1_v2.h5



Epoch 6: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 29s 1s/step - accuracy: 0.5970 - loss: 1.3265 - val_accuracy: 0.6244 - val_loss: 1.3765 - learning_rate: 5.0000e-04
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 993ms/step - accuracy: 0.6224 - loss: 1.2283
Epoch 7: val_accuracy improved from 0.62439 to 0.63902, saving model to best_model_phase1_v2.h5



Epoch 7: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step - accuracy: 0.6356 - loss: 1.2195 - val_accuracy: 0.6390 - val_loss: 1.2894 - learning_rate: 5.0000e-04
Epoch 8/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 914ms/step - accuracy: 0.7059 - loss: 1.0712
Epoch 8: val_accuracy improved from 0.63902 to 0.65854, saving model to best_model_phase1_v2.h5


... (6 outputs left out)


Epoch 20: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.8284 - loss: 0.5481 - val_accuracy: 0.7415 - val_loss: 0.9114 - learning_rate: 5.0000e-04
Epoch 21/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 940ms/step - accuracy: 0.8266 - loss: 0.5362
Epoch 21: val_accuracy improved from 0.74146 to 0.75122, saving model to best_model_phase1_v2.h5



Epoch 21: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.8259 - loss: 0.5601 - val_accuracy: 0.7512 - val_loss: 0.9748 - learning_rate: 5.0000e-04
Epoch 22/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 937ms/step - accuracy: 0.8636 - loss: 0.4949
Epoch 22: val_accuracy did not improve from 0.75122
26/26 ━━━━━━━━━━━━━━━━━━━━ 29s 1s/step - accuracy: 0.8607 - loss: 0.5021 - val_accuracy: 0.7317 - val_loss: 0.9793 - learning_rate: 5.0000e-04
Epoch 23/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 971ms/step - accuracy: 0.8687 - loss: 0.4468
Epoch 23: val_accuracy did not improve from 0.75122
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.8371 - loss: 0.5237 - val_accuracy: 0.7171 - val_loss: 0.9557 - learning_rate: 5.0000e-04
Epoch 24/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 940ms/step - accuracy: 0.8371 - loss: 0.5482
Epoch 24: val_accuracy did not improve from 0.75122
26/26 ━━━━━━━━━━━━━━━━━━━━ 29s 1s/step - accuracy: 0.8520 - loss: 0.4876 - val_accuracy: 0.722


Epoch 40: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.8843 - loss: 0.3624 - val_accuracy: 0.7561 - val_loss: 0.9492 - learning_rate: 1.2500e-04
Epoch 41/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 943ms/step - accuracy: 0.9097 - loss: 0.3035
Epoch 41: val_accuracy did not improve from 0.75610
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.9179 - loss: 0.2805 - val_accuracy: 0.7415 - val_loss: 0.9485 - learning_rate: 1.2500e-04
Epoch 42/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 960ms/step - accuracy: 0.9033 - loss: 0.3509
Epoch 42: val_accuracy did not improve from 0.75610
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.9129 - loss: 0.3087 - val_accuracy: 0.7512 - val_loss: 0.9582 - learning_rate: 1.2500e-04
Epoch 43/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 797ms/step - accuracy: 0.8961 - loss: 0.3289
Epoch 43: val_accuracy did not improve from 0.75610
26/26 ━━━━━━━━━━━━━━━━━━━━ 25s 967ms/step - accuracy: 0.9030 - loss: 0.3036 - val_accuracy: 0.


Epoch 44: finished saving model to best_model_phase1_v2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.9067 - loss: 0.3125 - val_accuracy: 0.7707 - val_loss: 0.9354 - learning_rate: 1.2500e-04
Epoch 45/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 892ms/step - accuracy: 0.9062 - loss: 0.2966
Epoch 45: val_accuracy did not improve from 0.77073
26/26 ━━━━━━━━━━━━━━━━━━━━ 28s 1s/step - accuracy: 0.9104 - loss: 0.3033 - val_accuracy: 0.7707 - val_loss: 0.9346 - learning_rate: 1.2500e-04
Epoch 46/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 869ms/step - accuracy: 0.9096 - loss: 0.2993
Epoch 46: val_accuracy did not improve from 0.77073
26/26 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.9005 - loss: 0.3263 - val_accuracy: 0.7561 - val_loss: 0.9630 - learning_rate: 1.2500e-04
Epoch 47/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 833ms/step - accuracy: 0.9154 - loss: 0.3306
Epoch 47: val_accuracy did not improve from 0.77073
26/26 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.9030 - loss: 0.3487 - val_accuracy: 0.761

In [9]:
# ✅ FINE-TUNING
print("\n🔧 PHASE 2 : Fine-tuning...\n")

base_model.trainable = True

# Geler toutes les couches sauf les 20 dernières
for layer in base_model.layers[:-20]:
    layer.trainable = False

trainable_count = sum([1 for layer in base_model.layers if layer.trainable])
print(f"✅ Couches entraînables : {trainable_count}/{len(base_model.layers)}")

# RECOMPILER avec LR très bas
model.compile(
    optimizer=Adam(learning_rate=0.00001),  # 10x plus bas
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# CALLBACKS
callbacks_ft = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=12,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.5,
        patience=4,
        min_lr=1e-8,
        mode='max',
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_phase2.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# ENTRAÎNEMENT
history_ft = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=callbacks_ft,
    verbose=1
)

# RÉSULTATS
best_val_acc_ft = max(history_ft.history['val_accuracy'])
print(f"\n✅ PHASE 2 TERMINÉE !")
print(f"✅ Meilleure val_accuracy après fine-tuning : {best_val_acc_ft*100:.2f}%")


🔧 PHASE 2 : Fine-tuning...

✅ Couches entraînables : 20/154
Epoch 1/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7841 - loss: 0.7557
Epoch 1: val_accuracy improved from None to 0.74146, saving model to best_model_phase2.h5



Epoch 1: finished saving model to best_model_phase2.h5
26/26 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.7799 - loss: 0.7213 - val_accuracy: 0.7415 - val_loss: 0.9242 - learning_rate: 1.0000e-05
Epoch 2/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8154 - loss: 0.6718
Epoch 2: val_accuracy did not improve from 0.74146
26/26 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.7960 - loss: 0.6773 - val_accuracy: 0.7122 - val_loss: 0.9271 - learning_rate: 1.0000e-05
Epoch 3/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7933 - loss: 0.7202
Epoch 3: val_accuracy did not improve from 0.74146
26/26 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.7898 - loss: 0.6967 - val_accuracy: 0.7171 - val_loss: 0.9376 - learning_rate: 1.0000e-05
Epoch 4/25
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7783 - loss: 0.7375
Epoch 4: val_accuracy did not improve from 0.74146
26/26 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step - accuracy: 0.7749 - loss: 0.7142 - val_accuracy: 0.7122 - val_loss: 0.950

In [10]:
# ✅ SAUVEGARDER
model.save("landmark_model_final.h5")
print("✅ Modèle final sauvegardé : landmark_model_final.h5")

# ✅ TFLITE (pour mobile)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("landmark_model.tflite", "wb") as f:
    f.write(tflite_model)
    
print("✅ Modèle TFLite sauvegardé : landmark_model.tflite")

✅ Modèle final sauvegardé : landmark_model_final.h5
INFO:tensorflow:Assets written to: C:\Users\WALID\AppData\Local\Temp\tmp1gok2piv\assets


INFO:tensorflow:Assets written to: C:\Users\WALID\AppData\Local\Temp\tmp1gok2piv\assets


Saved artifact at 'C:\Users\WALID\AppData\Local\Temp\tmp1gok2piv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 21), dtype=tf.float32, name=None)
Captures:
  3040164095824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164100048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164101968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164101584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164100432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164101776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164100240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164101392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164101200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040164100624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  3040

In [53]:
import tensorflow as tf
import numpy as np
import json

# 1. Charger le modèle utilisé par Streamlit
print("1️⃣ Chargement du modèle de Streamlit...\n")
model = tf.keras.models.load_model(r'C:\Users\WALID\pfe_idsd\best_model_phase1_v2.h5')

# 2. Charger les classes
with open(r'C:\Users\WALID\pfe_idsd\class_names.json', 'r', encoding='utf-8') as f:
    class_names = json.load(f)

print(f"✅ Modèle chargé")
print(f"✅ {len(class_names)} classes\n")

# 3. Tester sur le validation set
print("2️⃣ Test sur validation set...\n")

val_data.reset()
correct = 0
total = 0

for i in range(len(val_data)):
    images, labels = val_data[i]
    predictions = model.predict(images, verbose=0)
    
    pred_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(labels, axis=1)
    
    correct += np.sum(pred_classes == true_classes)
    total += len(images)

accuracy = correct / total

print(f"📊 Accuracy sur validation : {accuracy*100:.2f}%")
print(f"✅ Correct : {correct}/{total}")
print(f"❌ Erreurs : {total - correct}")

if accuracy < 0.70:
    print("\n⚠️ ATTENTION : Accuracy inférieure à 70% !")
    print("   Le modèle chargé n'est peut-être pas le bon")
elif accuracy > 0.73 and accuracy < 0.74:
    print("\n✅ CORRECT : C'est bien le modèle à 73.66%")
else:
    print(f"\n⚠️ Accuracy différente de 73.66% : {accuracy*100:.2f}%")

1️⃣ Chargement du modèle de Streamlit...



✅ Modèle chargé
✅ 21 classes

2️⃣ Test sur validation set...

📊 Accuracy sur validation : 77.07%
✅ Correct : 158/205
❌ Erreurs : 47

⚠️ Accuracy différente de 73.66% : 77.07%


In [55]:
from PIL import Image
import numpy as np

print("\n3️⃣ Simulation du prétraitement Streamlit...\n")

# Prendre une image de validation
val_data.reset()
test_images, test_labels = next(val_data)
test_img = test_images[0]
true_label_idx = np.argmax(test_labels[0])
true_label = class_names[true_label_idx]

print(f"📸 Vraie classe : {true_label}\n")

# MÉTHODE 1 : Prédiction directe (comme dans le notebook)
pred1 = model.predict(np.expand_dims(test_img, axis=0), verbose=0)[0]
pred1_idx = np.argmax(pred1)
pred1_label = class_names[pred1_idx]
conf1 = pred1[pred1_idx]

print(f"1️⃣ Prédiction directe (notebook) :")
print(f"   → {pred1_label} ({conf1*100:.2f}%)")

# MÉTHODE 2 : Simulation exacte de Streamlit
# Convertir en PIL Image (comme uploadé dans Streamlit)
test_img_pil = Image.fromarray((test_img * 255).astype('uint8'))

# Appliquer le même prétraitement que Streamlit
if test_img_pil.mode != 'RGB':
    test_img_pil = test_img_pil.convert('RGB')

test_img_pil = test_img_pil.resize((224, 224))
img_array = np.array(test_img_pil) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Prédire
pred2 = model.predict(img_array, verbose=0)[0]
pred2_idx = np.argmax(pred2)
pred2_label = class_names[pred2_idx]
conf2 = pred2[pred2_idx]

print(f"\n2️⃣ Prédiction Streamlit simulée :")
print(f"   → {pred2_label} ({conf2*100:.2f}%)")

print(f"\n3️⃣ Comparaison :")
print(f"   Vraie classe : {true_label}")
print(f"   Prédiction 1 : {pred1_label} ({'✅' if pred1_label == true_label else '❌'})")
print(f"   Prédiction 2 : {pred2_label} ({'✅' if pred2_label == true_label else '❌'})")

if pred1_label != pred2_label:
    print("\n⚠️ PROBLÈME DÉTECTÉ : Les deux méthodes donnent des résultats différents !")
    print("   → Problème de prétraitement")
else:
    print("\n✅ Les deux méthodes donnent le même résultat")


3️⃣ Simulation du prétraitement Streamlit...

📸 Vraie classe : Almoravid_Koubba

1️⃣ Prédiction directe (notebook) :
   → Almoravid_Koubba (99.01%)

2️⃣ Prédiction Streamlit simulée :
   → Almoravid_Koubba (99.01%)

3️⃣ Comparaison :
   Vraie classe : Almoravid_Koubba
   Prédiction 1 : Almoravid_Koubba (✅)
   Prédiction 2 : Almoravid_Koubba (✅)

✅ Les deux méthodes donnent le même résultat


In [57]:
print("\n4️⃣ Vérification de l'ordre des classes...\n")

# Ordre dans class_names.json
print("📋 Ordre dans class_names.json (utilisé par Streamlit) :")
for i, name in enumerate(class_names[:5]):  # Afficher les 5 premiers
    print(f"   {i}: {name}")
print("   ...")

# Ordre dans train_data
print("\n📋 Ordre dans train_data.class_indices :")
train_indices = sorted(train_data.class_indices.items(), key=lambda x: x[1])
for i, (name, idx) in enumerate(train_indices[:5]):  # Afficher les 5 premiers
    print(f"   {idx}: {name}")
print("   ...")

# Comparer
class_names_from_train = [name for name, idx in train_indices]

if class_names == class_names_from_train:
    print("\n✅ Ordres IDENTIQUES - Pas de problème")
else:
    print("\n❌ ATTENTION : Ordres DIFFÉRENTS !")
    print("\n🔧 Différences détectées :")
    for i, (json_name, train_name) in enumerate(zip(class_names, class_names_from_train)):
        if json_name != train_name:
            print(f"   Position {i}: '{json_name}' ≠ '{train_name}'")


4️⃣ Vérification de l'ordre des classes...

📋 Ordre dans class_names.json (utilisé par Streamlit) :
   0: Almoravid_Koubba
   1: chellah
   2: Jardin_Majorelle
   3: kasbah d'agadir oufella
   4: Kasbah_Amridil
   ...

📋 Ordre dans train_data.class_indices :
   0: Almoravid_Koubba
   1: Jardin_Majorelle
   2: Kasbah_Amridil
   3: Kasbah_de_Beni_Mellal
   4: Kasbah_de_Taourirt
   ...

❌ ATTENTION : Ordres DIFFÉRENTS !

🔧 Différences détectées :
   Position 1: 'chellah' ≠ 'Jardin_Majorelle'
   Position 2: 'Jardin_Majorelle' ≠ 'Kasbah_Amridil'
   Position 3: 'kasbah d'agadir oufella' ≠ 'Kasbah_de_Beni_Mellal'
   Position 4: 'Kasbah_Amridil' ≠ 'Kasbah_de_Taourirt'
   Position 5: 'Kasbah_de_Beni_Mellal' ≠ 'Kasbah_des_Oudayas'
   Position 6: 'Kasbah_de_Taourirt' ≠ 'Ksar_d’Aït_Ben_Haddou'
   Position 7: 'Kasbah_des_Oudayas' ≠ 'Mausolee_de_Mohammed V'
   Position 8: 'Ksar_d’Aït_Ben_Haddou' ≠ 'Mosquee_Hassan II'
   Position 9: 'Mausolee_de_Mohammed V' ≠ 'Mosquee_Koutoubia'
   Position 10: 'men

In [65]:
import os
import json

train_dir = "C:\\Users\\WALID\\pfe_idsd\\train"

class_names = sorted(os.listdir(train_dir))

with open("C:\\Users\\WALID\\pfe_idsd\\class_names.json", "w") as f:
    json.dump(class_names, f)

print(class_names)


['Almoravid_Koubba', 'Jardin_Majorelle', 'Kasbah_Amridil', 'Kasbah_de_Beni_Mellal', 'Kasbah_de_Taourirt', 'Kasbah_des_Oudayas', 'Ksar_d’Aït_Ben_Haddou', 'Mausolee_de_Mohammed V', 'Mosquee_Hassan II', 'Mosquee_Koutoubia', 'Mosquee_Tinmal', 'Mosquee_de_la_Kasbah', 'Palais_El_Badi', 'Palais_de_la_Bahia', 'Place_Jemaa_el-Fna', 'Tombeaux_Saadiens', 'Volubilis meknès', 'chellah', "kasbah d'agadir oufella", 'menara', 'palais royal de fès']


In [67]:
print("Sorties modèle :", model.output_shape)
print("Nombre classes JSON :", len(class_names))


Sorties modèle : (None, 21)
Nombre classes JSON : 21
